# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [14]:
# imports
import os
import json
from IPython.display import Markdown, display, clear_output
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [15]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [16]:
# Initialize openAI
openai = OpenAI(base_url='http://127.0.0.1:11434/v1', api_key='ollama')

In [17]:
#system prompt
system_prompt = """
You are a senior developer but you are acting as a junior just so the user which is here, a fresher who is learning to build something finally can ask you freely as if you are one of them instead of guiding him like a senior. Your answer will show your experience but your tone will show simplicity.
"""

In [18]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why, what if we do not use it then what could be the alternative:
const generateToken = (id) => {
  if (!process.env.JWT_SECRET) {
    throw new Error("JWT_SECRET is missing from environment variables.");
  }
  return jwt.sign({ id }, process.env.JWT_SECRET, { expiresIn: "7d" });
};
"""

In [19]:
# Get gpt-4o-mini to answer, with streaming
stream = openai.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[{"role":"system", "content": system_prompt},{
        "role": "user", "content":question
    }],
    stream=True
)

answer = ""
display_handle = display(Markdown("Thinking..."), display_id=True)

for chunk in stream:
    content = chunk.choices[0].delta.content or ""
    answer += content
    clear_output(wait=True)
    display(Markdown(answer))

So this code generates a JSON Web Token (JWT). 

**What does it do?**

It takes an `id` as an argument and returns a token that can be used to identify the user. Think of it like a unique string that contains some personal info (in this case, just `id`) but also has a expiration date.

Here's how it works:

- The function first checks if there's a secret code available in environment variables (`process.env.JWT_SECRET`).
- If not, it throws an error because the token can't be generated without knowing the secret.
- After that, it uses the `jwt.sign()` method to sign the `id` with the JWT secret and with an expiration time of 7 days.

**Why is this code necessary?**

It's usually used in web applications when you want to store some kind of data for the user, like permissions or information about their profile. 

When a new request comes to your server from an authenticated user (with the token), you can verify that the token is correct and use its date of expiration.

**What if we don't use it then what could be the alternative?**

One way we won't need JWT is when our server application doesn't require session management, i.e., not storing data in the user's browser or verifying some permissions for each request. 

In such a case, cookies can also be used to store info.

Here's how this code would look like using Cookies:

```javascript
// Set a cookie with id and expiration set to 7 days.
function generateToken(id) {
    if (!process.env.JWT_SECRET) {
        throw new Error("JWT_SECRET is missing from environment variables.");
    }
    const cookieOptions = { secure: process.env.NODE_ENV !== 'development', HttpOnly: true };
    
    // Set expiration in cookies
    const cookieExpireDate = Math.floor(Date.now() / 1000) + (7 * 24 * 60 * 60);
    document.cookie = `id=${id}; expiration="${new Date(cookieExpireDate).toUTCString()}"; ${JSON.stringify(cookieOptions)}`;
    return '';
}

// To use it, you can do
function verifyId(token) {
    const cookieValue = getCookie('id');
    // Verify token here, you might need a library for this as well
}
```

This way we have to remember about sessions though which will make things more complex.